In [1]:
# Importing Libraries
import pandas as pd
import sqlite3
import json
import os

DB_PATH = "../data/patents.db"
REPORTS_DIR = "../reports/"

conn = sqlite3.connect(DB_PATH)

print("Connected to database")
print("Libraries loaded")

Connected to database
Libraries loaded


In [2]:
# Loading all report data
df_inventors = pd.read_csv(REPORTS_DIR + "top_inventors.csv")
df_companies = pd.read_csv(REPORTS_DIR + "top_companies.csv")
df_countries = pd.read_csv(REPORTS_DIR + "country_trends.csv")
df_years = pd.read_csv(REPORTS_DIR + "patents_per_year.csv")

total_patents = pd.read_sql("SELECT COUNT(*) as total FROM patents", conn).iloc[0]['total']
total_inventors = pd.read_sql("SELECT COUNT(DISTINCT inventor_id) as total FROM inventors", conn).iloc[0]['total']
total_companies = pd.read_sql("SELECT COUNT(DISTINCT company_id) as total FROM companies", conn).iloc[0]['total']
total_countries = pd.read_sql("SELECT COUNT(DISTINCT country) as total FROM inventors WHERE country IS NOT NULL", conn).iloc[0]['total']

print("All data loaded")

All data loaded


In [3]:
# Console Report
print("")
print("GLOBAL PATENT INTELLIGENCE REPORT")
print("-" * 55)

print(f"\nTOTAL PATENTS:    {total_patents:,}")
print(f"TOTAL INVENTORS:  {total_inventors:,}")
print(f"TOTAL COMPANIES:  {total_companies:,}")
print(f"TOTAL COUNTRIES:  {total_countries:,}")

print("\n" + " " )
print("TOP 10 INVENTORS BY PATENT COUNT")
print("-" * 55)
for i, row in df_inventors.head(10).iterrows():
    print(f"{i+1:>3}. {row['full_name']:<35} {row['patent_count']:>5} patents  ({row['country']})")

print("\n" + "-" * 55)
print("TOP 10 COMPANIES BY PATENT COUNT")
print("-" * 55)
for i, row in df_companies.head(10).iterrows():
    print(f"{i+1:>3}. {row['name']:<50} {row['patent_count']:>5} patents")

print("\n" + "-" * 55)
print("TOP 10 COUNTRIES BY PATENT COUNT")
print("-" * 55)
for i, row in df_countries.head(10).iterrows():
    print(f"{i+1:>3}. {row['country']:<10} {row['patent_count']:>8,} patents")

print("\n" + "")
print("PATENTS BY YEAR")
print("-" * 55)
for i, row in df_years.iterrows():
    print(f"     {int(row['year'])}: {row['patent_count']:,} patents")




GLOBAL PATENT INTELLIGENCE REPORT
-------------------------------------------------------

TOTAL PATENTS:    100,000
TOTAL INVENTORS:  92,442
TOTAL COMPANIES:  31,089
TOTAL COUNTRIES:  109

 
TOP 10 INVENTORS BY PATENT COUNT
-------------------------------------------------------
  1. Shunpei Yamazaki                       36 patents  (JP)
  2. Kia Silverbrook                        23 patents  (AU)
  3. Tao Luo                                16 patents  (US)
  4. Junyi Li                               13 patents  (US)
  5. Bartley K. Andre                       12 patents  (US)
  6. Matthew Dean Rohrbach                  11 patents  (US)
  7. Jing Sun                               11 patents  (US)
  8. Gurtej S. Sandhu                       11 patents  (US)
  9. Duncan Robert Kerr                     11 patents  (US)
 10. Roderick A. Hyde                       10 patents  (US)

-------------------------------------------------------
TOP 10 COMPANIES BY PATENT COUNT
------------------

In [4]:
# JSON Report
total_country_patents = df_countries['patent_count'].sum()

report = {
    "report_title": "Global Patent Intelligence Report",
    "data_source": "PatentsView - USPTO",
    "summary": {
        "total_patents": int(total_patents),
        "total_inventors": int(total_inventors),
        "total_companies": int(total_companies),
        "total_countries": int(total_countries)
    },
    "top_inventors": [
        {
            "rank": i + 1,
            "name": row['full_name'],
            "country": row['country'],
            "patents": int(row['patent_count'])
        }
        for i, row in df_inventors.head(10).iterrows()
    ],
    "top_companies": [
        {
            "rank": i + 1,
            "name": row['name'],
            "patents": int(row['patent_count'])
        }
        for i, row in df_companies.head(10).iterrows()
    ],
    "top_countries": [
        {
            "rank": i + 1,
            "country": row['country'],
            "patents": int(row['patent_count']),
            "share": round(row['patent_count'] / total_country_patents, 4)
        }
        for i, row in df_countries.head(10).iterrows()
    ],
    "patents_by_year": [
        {
            "year": int(row['year']),
            "patents": int(row['patent_count'])
        }
        for i, row in df_years.iterrows()
    ]
}

with open(REPORTS_DIR + "report.json", "w") as f:
    json.dump(report, f, indent=4)

print("report.json saved!")
print(json.dumps(report, indent=4))

report.json saved!
{
    "report_title": "Global Patent Intelligence Report",
    "data_source": "PatentsView - USPTO",
    "summary": {
        "total_patents": 100000,
        "total_inventors": 92442,
        "total_companies": 31089,
        "total_countries": 109
    },
    "top_inventors": [
        {
            "rank": 1,
            "name": "Shunpei Yamazaki",
            "country": "JP",
            "patents": 36
        },
        {
            "rank": 2,
            "name": "Kia Silverbrook",
            "country": "AU",
            "patents": 23
        },
        {
            "rank": 3,
            "name": "Tao Luo",
            "country": "US",
            "patents": 16
        },
        {
            "rank": 4,
            "name": "Junyi Li",
            "country": "US",
            "patents": 13
        },
        {
            "rank": 5,
            "name": "Bartley K. Andre",
            "country": "US",
            "patents": 12
        },
        {
            "r

In [5]:
# Verifying all report files exist
files = [
    "top_inventors.csv",
    "top_companies.csv",
    "country_trends.csv",
    "patents_per_year.csv",
    "report.json"
]

print("REPORT FILES CHECK")
print("-" * 40)
for f in files:
    path = REPORTS_DIR + f
    exists = os.path.exists(path)
    print(f"{f}: {'EXISTS' if exists else 'MISSING'}")

REPORT FILES CHECK
----------------------------------------
top_inventors.csv: EXISTS
top_companies.csv: EXISTS
country_trends.csv: EXISTS
patents_per_year.csv: EXISTS
report.json: EXISTS


In [6]:
#Closing Connection
conn.close()
print("Database connection closed")


Database connection closed
